<a href="https://colab.research.google.com/github/heyitsarun/Aero-Gen-Synthetic-Anomaly-Generation-in-Aviation-Communications/blob/main/02_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/aerogen_project'
DATA_DIR = f'{PROJECT_ROOT}/data'
GEN_DIR = f'{PROJECT_ROOT}/generated'
os.makedirs(GEN_DIR, exist_ok=True)

df = pd.read_csv(f'{DATA_DIR}/atcosim_enriched.csv')
asrs_df = pd.read_csv(f'{DATA_DIR}/asrs_clean_taxonomy.csv')
print(f'Loaded {len(df)} enriched ATCOSIM rows, {len(asrs_df)} ASRS taxonomy rows')

Mounted at /content/drive
Loaded 6519 enriched ATCOSIM rows, 45 ASRS taxonomy rows


In [2]:
NUMBER_WORDS_LIST = ['zero','one','two','three','four','five','six','seven','eight','nine']
TRAILING_STRIP_WORDS = {'radar', 'control', 'approach', 'departure', 'tower', 'ground', 'in', 'is'}
LEADING_STRIP_WORDS = {'break'}

def clean_callsign(callsign):
    tokens = callsign.split()
    while tokens and tokens[0] in LEADING_STRIP_WORDS:
        tokens = tokens[1:]
    while tokens and tokens[-1] in TRAILING_STRIP_WORDS:
        tokens = tokens[:-1]
    return ' '.join(tokens)

def extract_airline_and_number(callsign):
    tokens = callsign.split()
    airline_tokens, number_tokens = [], []
    seen_number = False
    for tok in tokens:
        if tok in NUMBER_WORDS_LIST:
            seen_number = True
            number_tokens.append(tok)
        elif not seen_number:
            airline_tokens.append(tok)
        else:
            number_tokens.append(tok)
    return ' '.join(airline_tokens), number_tokens

df['callsign'] = df['callsign'].apply(clean_callsign)
df[['airline', 'flight_number_tokens']] = df['callsign'].apply(
    lambda c: pd.Series(extract_airline_and_number(c))
)

CONFIRMED_AIRLINE_THRESHOLD = 5
airline_frequency = df['airline'].value_counts()
confirmed_airlines = set(airline_frequency[airline_frequency >= CONFIRMED_AIRLINE_THRESHOLD].index)

valid_swap_pool = df[
    df['airline'].isin(confirmed_airlines) &
    df['flight_number_tokens'].apply(lambda toks: any(t in NUMBER_WORDS_LIST for t in toks))
].reset_index(drop=True)

print(f'{len(confirmed_airlines)} confirmed airlines, {len(valid_swap_pool)} rows in final swap pool')

134 confirmed airlines, 5679 rows in final swap pool


In [3]:
def corrupt_same_company_similar_number(row, swap_pool):
    airline = row['airline']
    same_airline_rows = swap_pool[(swap_pool['airline'] == airline) & (swap_pool['callsign'] != row['callsign'])]
    if len(same_airline_rows) == 0:
        return None
    swap_target = same_airline_rows.sample(1).iloc[0]
    corrupted_callsign = swap_target['callsign']
    corrupted_text = row['text'].replace(row['callsign'], corrupted_callsign, 1)
    return {'anomaly_type': 'callsign_confusion_same_company', 'original_text': row['text'],
            'corrupted_text': corrupted_text, 'original_callsign': row['callsign'],
            'corrupted_callsign': corrupted_callsign, 'source_pattern': 'ASRS: same-company similar flight number'}

def corrupt_similar_sounding_different_operator(row, swap_pool):
    other_airlines = swap_pool[swap_pool['airline'] != row['airline']]['airline'].unique()
    if len(other_airlines) == 0:
        return None
    target_airline = random.choice(other_airlines)
    swap_target = swap_pool[swap_pool['airline'] == target_airline].sample(1).iloc[0]
    corrupted_callsign = swap_target['callsign']
    corrupted_text = row['text'].replace(row['callsign'], corrupted_callsign, 1)
    return {'anomaly_type': 'callsign_confusion_different_operator', 'original_text': row['text'],
            'corrupted_text': corrupted_text, 'original_callsign': row['callsign'],
            'corrupted_callsign': corrupted_callsign, 'source_pattern': 'ASRS: similar-sounding callsign, different operator'}

def corrupt_no_callsign_used(row):
    if not row['instruction']:
        return None
    return {'anomaly_type': 'no_callsign_used', 'original_text': row['text'],
            'corrupted_text': row['instruction'], 'original_callsign': row['callsign'],
            'corrupted_callsign': '', 'source_pattern': 'ASRS: instruction issued without stating callsign'}

def corrupt_readback_digit_error(row):
    tokens = row['instruction'].split()
    number_positions = [i for i, t in enumerate(tokens) if t in NUMBER_WORDS_LIST]
    if not number_positions:
        return None
    pos = random.choice(number_positions)
    new_digit = random.choice([d for d in NUMBER_WORDS_LIST if d != tokens[pos]])
    corrupted_tokens = tokens.copy()
    corrupted_tokens[pos] = new_digit
    corrupted_instruction = ' '.join(corrupted_tokens)
    corrupted_text = row['text'].replace(row['instruction'], corrupted_instruction, 1)
    return {'anomaly_type': f"readback_error_{row['value_type']}", 'original_text': row['text'],
            'corrupted_text': corrupted_text, 'original_callsign': row['callsign'],
            'corrupted_callsign': row['callsign'], 'source_pattern': f"ASRS: digit misread in {row['value_type']} readback"}

In [4]:
RULE_BASED_PATH = f'{GEN_DIR}/synthetic_rule_based.csv'
synthetic_df = pd.read_csv(RULE_BASED_PATH)
print(f'Loaded existing rule-based set: {len(synthetic_df)} rows')
print(synthetic_df['anomaly_type'].value_counts())

Loaded existing rule-based set: 1312 rows
anomaly_type
no_callsign_used                         286
callsign_confusion_different_operator    272
readback_error_altitude                  232
readback_error_frequency                 229
callsign_confusion_same_company          202
readback_error_heading                    46
readback_error_squawk_code                27
readback_error_unclassified               18
Name: count, dtype: int64


In [5]:
LLM_OUTPUT_PATH = f'{GEN_DIR}/synthetic_llm_generated.csv'
llm_synthetic_df = pd.read_csv(LLM_OUTPUT_PATH)
before = len(llm_synthetic_df)

llm_synthetic_df = llm_synthetic_df[
    llm_synthetic_df['anomaly_type'] != 'readback_error_altitude'
].reset_index(drop=True)

print(f'Removed {before - len(llm_synthetic_df)} readback_error_altitude rows (pre-fix, unreliable)')
llm_synthetic_df.to_csv(LLM_OUTPUT_PATH, index=False)
print(llm_synthetic_df['anomaly_type'].value_counts())

Removed 0 readback_error_altitude rows (pre-fix, unreliable)
anomaly_type
callsign_confusion_same_company          60
callsign_confusion_different_operator    60
no_callsign_used                         60
missing_similarity_warning               60
Name: count, dtype: int64


## Part B — LLM Few-Shot Generator
Uses the ASRS taxonomy narratives as grounding examples to generate anomalies in ATCOSIM-style phraseology, covering the same anomaly types as the rule-based baseline plus the "missing similarity warning" pattern.

In [6]:
!pip install -q groq

from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print(response.choices[0].message.content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.7 MB/s eta 0:00:00
Hello, it's nice to meet you and I'm looking forward to chatting with you.


Using Groq as it is the only LLM that offers free tier.

In [7]:
cc_examples = asrs_df[asrs_df['Category'].isin(['CC', 'BOTH'])]['Synopsis'].dropna().sample(4, random_state=1).tolist()
rb_examples = asrs_df[asrs_df['Category'].isin(['RB', 'BOTH'])]['Synopsis'].dropna().sample(2, random_state=1).tolist()

SYSTEM_PROMPT = """You generate synthetic Air Traffic Control (ATC) transcript pairs for a safety-research dataset.

STYLE RULES (must match exactly):
- All lowercase, no punctuation except decimal points spoken as "decimal"
- Numbers spelled out as words (e.g. "one three four" not "134")
- Format: "<callsign> <instruction>" — matches real ATCOSIM corpus phraseology
- Keep each line under 20 words, realistic ATC radiotelephony style
- When substituting an airline/callsign, choose a plausible, VARIED commercial airline different from the original

You will be given a NORMAL transcript line and an ANOMALY TYPE. Produce a CORRUPTED version of that line reflecting the anomaly type, grounded in the real incident pattern described.

Respond with ONLY the corrupted transcript line. No explanation, no quotes, no extra text.
"""

ANOMALY_TYPE_GROUNDING = {
    'callsign_confusion_same_company': f"Real incident pattern: {cc_examples[0]}",
    'callsign_confusion_different_operator': f"Real incident pattern: {cc_examples[1]}",
    'no_callsign_used': f"Real incident pattern: {cc_examples[2]}",
    'missing_similarity_warning': f"Real incident pattern: {cc_examples[3]} — ATC failed to flag a known similar-callsign pair before issuing the instruction.",
    'readback_error_altitude': f"Real incident pattern: {rb_examples[0]}",
    'readback_error_frequency': f"Real incident pattern: {rb_examples[1]}",
    'readback_error_heading': f"Real incident pattern: {rb_examples[0]} — applied to a heading readback instead of altitude.",
    'readback_error_squawk_code': f"Real incident pattern: {rb_examples[1]} — applied to a squawk code readback instead of frequency.",
}

READBACK_VALUE_MAP = {
    'readback_error_altitude': 'altitude', 'readback_error_frequency': 'frequency',
    'readback_error_heading': 'heading', 'readback_error_squawk_code': 'squawk_code',
}

def get_sample_source(anomaly_type, df_ref):
    if anomaly_type in READBACK_VALUE_MAP:
        target = READBACK_VALUE_MAP[anomaly_type]
        filtered = df_ref[(df_ref['parser_confidence'] == 'high') & (df_ref['value_type'] == target)]
        if len(filtered) > 0:
            return filtered.sample(frac=1, random_state=7).reset_index(drop=True)
    return df_ref[df_ref['parser_confidence'] == 'high'].sample(frac=1, random_state=7).reset_index(drop=True)

def generate_llm_anomaly(row, anomaly_type, client, max_retries=3):
    grounding = ANOMALY_TYPE_GROUNDING.get(anomaly_type, "")
    user_prompt = f"""NORMAL transcript: "{row['text']}"
ANOMALY TYPE: {anomaly_type}
{grounding}

Produce the corrupted version."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
              model="llama-3.1-8b-instant",  # was: llama-3.3-70b-versatile
              messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}],
              temperature=0.9, max_tokens=60
            )
            return response.choices[0].message.content.strip().strip('"').lower()
        except Exception as e:
            wait = 190 if '429' in str(e) else 2 ** attempt
            if attempt < max_retries - 1:
                time.sleep(wait)
            else:
                print(f"Failed after {max_retries} attempts: {e}")
                return None

In [9]:
sample_source = get_sample_source('readback_error_altitude', df)
test_row = sample_source.iloc[0]

result = generate_llm_anomaly(test_row, 'readback_error_altitude', client)

print("Original: ", test_row['text'])
print("Corrupted:", result)

Original:  alitalia four zero four you are identified route ageri zurich east nelli maintain flight level three two zero
Corrupted: alitalia four zero four you are identified route ageri zurich east nelli cleared to flight level nine zero


In [10]:
REMAINING_TYPES = ['readback_error_altitude', 'readback_error_frequency', 'readback_error_heading', 'readback_error_squawk_code']
N_PER_TYPE = 60

already_done = llm_synthetic_df.groupby('anomaly_type').size().to_dict() if len(llm_synthetic_df) else {}
new_rows = []

for anomaly_type in REMAINING_TYPES:
    n_have = already_done.get(anomaly_type, 0)
    n_needed = max(0, N_PER_TYPE - n_have)
    print(f'{anomaly_type}: generating {n_needed} more (already have {n_have})...')

    sample_source = get_sample_source(anomaly_type, df)
    row_pointer = 0
    for _ in range(n_needed):
        row = sample_source.iloc[row_pointer % len(sample_source)]
        row_pointer += 1
        corrupted = generate_llm_anomaly(row, anomaly_type, client)
        if corrupted:
            new_rows.append({'anomaly_type': anomaly_type, 'original_text': row['text'],
                              'corrupted_text': corrupted, 'generation_method': 'llm_few_shot', 'label': 1})
        if len(new_rows) % 20 == 0 and new_rows:
            pd.concat([llm_synthetic_df, pd.DataFrame(new_rows)], ignore_index=True).to_csv(LLM_OUTPUT_PATH, index=False)
            print(f'  ...checkpoint saved ({len(new_rows)} new rows so far)')
        time.sleep(0.3)

llm_synthetic_df = pd.concat([llm_synthetic_df, pd.DataFrame(new_rows)], ignore_index=True)
llm_synthetic_df.to_csv(LLM_OUTPUT_PATH, index=False)
print(f'\nTotal LLM-generated rows: {len(llm_synthetic_df)}')
print(llm_synthetic_df['anomaly_type'].value_counts())

readback_error_altitude: generating 60 more (already have 0)...
  ...checkpoint saved (20 new rows so far)
  ...checkpoint saved (40 new rows so far)
  ...checkpoint saved (60 new rows so far)
readback_error_frequency: generating 60 more (already have 0)...
  ...checkpoint saved (80 new rows so far)
  ...checkpoint saved (100 new rows so far)
  ...checkpoint saved (120 new rows so far)
readback_error_heading: generating 60 more (already have 0)...
  ...checkpoint saved (140 new rows so far)
  ...checkpoint saved (160 new rows so far)
  ...checkpoint saved (180 new rows so far)
readback_error_squawk_code: generating 60 more (already have 0)...
  ...checkpoint saved (200 new rows so far)
  ...checkpoint saved (220 new rows so far)
  ...checkpoint saved (240 new rows so far)

Total LLM-generated rows: 480
anomaly_type
callsign_confusion_same_company          60
callsign_confusion_different_operator    60
no_callsign_used                         60
missing_similarity_warning               

In [11]:
for atype in ['readback_error_heading', 'readback_error_squawk_code']:
    print(f"=== {atype} ===")
    sample = llm_synthetic_df[llm_synthetic_df['anomaly_type'] == atype].sample(3, random_state=1)
    for _, r in sample.iterrows():
        print(f"  orig: {r['original_text']}")
        print(f"  corr: {r['corrupted_text']}")
        print()

=== readback_error_heading ===
  orig: crossair five one two turn right to willisau
  corr: crossair five one two turn right to willisau decimal two three zero decimal

  orig: lufthansa four six five two turn right heading two one zero
  corr: delta airlines four six five two turn right heading two three zero

  orig: fox sierra india turn left by ten degrees
  corr: fox sierra india turn left by twenty five degrees decimal decimal not decimal

=== readback_error_squawk_code ===
  orig: alitalia four zero four zurich radar good morning squawk seven five three two
  corr: alitalia four zero four zurich radar good morning squawk three eight one two

  orig: air france zero zero nine november squawk three four no two seven four four
  corr: alitalia zero two eight juliet squawk three four no two seven four four

  orig: air france three five six squawk seven five three six
  corr: american airlines three seven eight says five four six



In [12]:
for atype in ['readback_error_altitude', 'readback_error_frequency']:
    print(f"=== {atype} ===")
    sample = llm_synthetic_df[llm_synthetic_df['anomaly_type'] == atype].sample(3, random_state=2)
    for _, r in sample.iterrows():
        print(f"  orig: {r['original_text']}")
        print(f"  corr: {r['corrupted_text']}")
        print()

=== readback_error_altitude ===
  orig: air france six seven three climb to flight level three six zero
  corr: air bavaria four seven two climb to flight level eight zero

  orig: alitalia four zero four you are identified route ageri zurich east nelli maintain flight level three two zero
  corr: alitalia four zero four you are identified route ageri zurich east nelli maintain flight level nine zero

  orig: alitalia two nine two climb to flight level three one zero
  corr: alitalia two nine two climb to flight level eight zero decimal

=== readback_error_frequency ===
  orig: transwede one zero seven contact geneva one three three one five bye
  corr: transwede one zero seven contact transalpine one three five zero decimal

  orig: good morning emirates eight one radar contact proceed trasadingen zurich east fusse
  corr: good morning swiss nine two radar contact proceed trasadingen zurich east fusse

  orig: sabena eight three two contact rhein one two seven three seven tschuss
  co

In [13]:
# ============================================================
# STEP 1: Remove all previously-generated readback rows (had the callsign-drift bug)
# ============================================================
llm_synthetic_df = pd.read_csv(LLM_OUTPUT_PATH)
before = len(llm_synthetic_df)

llm_synthetic_df = llm_synthetic_df[
    ~llm_synthetic_df['anomaly_type'].str.startswith('readback_error')
].reset_index(drop=True)

print(f'Removed {before - len(llm_synthetic_df)} readback rows for regeneration')
print(llm_synthetic_df['anomaly_type'].value_counts())
print()

# ============================================================
# STEP 2: Redefine SYSTEM_PROMPT with the callsign-lock fix
# ============================================================
SYSTEM_PROMPT = """You generate synthetic Air Traffic Control (ATC) transcript pairs for a safety-research dataset.

STYLE RULES (must match exactly):
- All lowercase, no punctuation except decimal points spoken as "decimal"
- Numbers spelled out as words (e.g. "one three four" not "134")
- Format: "<callsign> <instruction>" — matches real ATCOSIM corpus phraseology
- Keep each line under 20 words, realistic ATC radiotelephony style
- When substituting an airline/callsign, choose a plausible, VARIED commercial airline different from the original

CRITICAL RULE: If the ANOMALY TYPE starts with "readback_error", you MUST keep the callsign EXACTLY unchanged — only change the single numeric value (altitude/frequency/heading/squawk code) mentioned in the instruction. Do not touch the callsign, do not rephrase the instruction verb, do not add extra words.

You will be given a NORMAL transcript line and an ANOMALY TYPE. Produce a CORRUPTED version of that line reflecting the anomaly type, grounded in the real incident pattern described.

Respond with ONLY the corrupted transcript line. No explanation, no quotes, no extra text.
"""

# ============================================================
# STEP 3: Redefine get_sample_source to require an actual number present
# ============================================================
def get_sample_source(anomaly_type, df_ref):
    if anomaly_type in READBACK_VALUE_MAP:
        target = READBACK_VALUE_MAP[anomaly_type]
        filtered = df_ref[
            (df_ref['parser_confidence'] == 'high') &
            (df_ref['value_type'] == target) &
            (df_ref['has_numbers'] == True)
        ]
        if len(filtered) > 0:
            return filtered.sample(frac=1, random_state=7).reset_index(drop=True)
    return df_ref[df_ref['parser_confidence'] == 'high'].sample(frac=1, random_state=7).reset_index(drop=True)

# ============================================================
# STEP 4: Run generation loop for all 4 readback types
# ============================================================
REMAINING_TYPES = ['readback_error_altitude', 'readback_error_frequency', 'readback_error_heading', 'readback_error_squawk_code']
N_PER_TYPE = 60

already_done = llm_synthetic_df.groupby('anomaly_type').size().to_dict() if len(llm_synthetic_df) else {}
new_rows = []

for anomaly_type in REMAINING_TYPES:
    n_have = already_done.get(anomaly_type, 0)
    n_needed = max(0, N_PER_TYPE - n_have)
    print(f'{anomaly_type}: generating {n_needed} more (already have {n_have})...')

    sample_source = get_sample_source(anomaly_type, df)
    row_pointer = 0
    for _ in range(n_needed):
        row = sample_source.iloc[row_pointer % len(sample_source)]
        row_pointer += 1
        corrupted = generate_llm_anomaly(row, anomaly_type, client)
        if corrupted:
            new_rows.append({'anomaly_type': anomaly_type, 'original_text': row['text'],
                              'corrupted_text': corrupted, 'generation_method': 'llm_few_shot', 'label': 1})
        if len(new_rows) % 20 == 0 and new_rows:
            pd.concat([llm_synthetic_df, pd.DataFrame(new_rows)], ignore_index=True).to_csv(LLM_OUTPUT_PATH, index=False)
            print(f'  ...checkpoint saved ({len(new_rows)} new rows so far)')
        time.sleep(0.3)

llm_synthetic_df = pd.concat([llm_synthetic_df, pd.DataFrame(new_rows)], ignore_index=True)
llm_synthetic_df.to_csv(LLM_OUTPUT_PATH, index=False)
print(f'\nTotal LLM-generated rows: {len(llm_synthetic_df)}')
print(llm_synthetic_df['anomaly_type'].value_counts())

# ============================================================
# STEP 5: Spot-check the result — confirm callsigns are now unchanged
# ============================================================
print('\n--- Spot check ---')
for atype in REMAINING_TYPES:
    print(f"=== {atype} ===")
    sample = llm_synthetic_df[llm_synthetic_df['anomaly_type'] == atype].sample(2, random_state=3)
    for _, r in sample.iterrows():
        print(f"  orig: {r['original_text']}")
        print(f"  corr: {r['corrupted_text']}")
    print()

Removed 240 readback rows for regeneration
anomaly_type
callsign_confusion_same_company          60
callsign_confusion_different_operator    60
no_callsign_used                         60
missing_similarity_warning               60
Name: count, dtype: int64

readback_error_altitude: generating 60 more (already have 0)...
  ...checkpoint saved (20 new rows so far)
  ...checkpoint saved (40 new rows so far)
  ...checkpoint saved (60 new rows so far)
readback_error_frequency: generating 60 more (already have 0)...
  ...checkpoint saved (80 new rows so far)
  ...checkpoint saved (100 new rows so far)
  ...checkpoint saved (120 new rows so far)
readback_error_heading: generating 60 more (already have 0)...
  ...checkpoint saved (140 new rows so far)
  ...checkpoint saved (160 new rows so far)
  ...checkpoint saved (180 new rows so far)
readback_error_squawk_code: generating 60 more (already have 0)...
  ...checkpoint saved (200 new rows so far)
  ...checkpoint saved (220 new rows so far)
  